# Lab 3: 语音合成 — Qwen3-TTS + OpenVINO

Qwen3-TTS 是阿里巴巴通义千问团队推出的最先进文本转语音模型。它支持多语言语音合成、声音克隆和声音设计等功能。

**主要特点:**
- 🌍 覆盖中、英、日、韩、德、法、俄、葡、西、意 10 大语言
- 🎭 支持多种预设说话人和风格指令控制
- ⚡ 基于双轨混合流式生成架构，首包延迟低至 97ms
- 🧠 智能文本理解，自适应调整语调、语速和情感表达

在本实验中，我们将：
1. 从 ModelScope 下载预转换的 OpenVINO 模型
2. 使用 OpenVINO 加载并运行语音合成
3. 体验交互式 Gradio 语音合成演示

#### 目录：
- [下载模型](#下载模型)
- [选择推理设备](#选择推理设备)
- [加载模型](#加载模型)
- [运行语音合成](#运行语音合成)
- [交互式演示](#交互式演示)

## 下载模型
[返回目录 ⬆️](#目录：)

从 ModelScope 下载已转换的 Qwen3-TTS-CustomVoice-0.6B OpenVINO 模型。如果模型已存在则跳过下载。

In [1]:
from pathlib import Path

model_dir = Path("Qwen3-TTS-CustomVoice-0.6B-fp16-ov")

if not model_dir.exists():
    from modelscope import snapshot_download
    snapshot_download("snake7gun/Qwen3-TTS-CustomVoice-0.6B-fp16-ov", local_dir=str(model_dir))
    print(f"模型已下载到: {model_dir}")
else:
    print(f"模型已存在: {model_dir}，跳过下载")

2026-04-22 17:13:02,347 - modelscope - INFO - Got 27 files, start to download ...


Processing 27 items:   0%|          | 0.00/27.0 [00:00<?, ?it/s]

2026-04-22 17:16:25,964 - modelscope - INFO - Finish downloading 27 files for repo 'snake7gun/Qwen3-TTS-CustomVoice-0.6B-fp16-ov'
2026-04-22 17:16:25,966 - modelscope - INFO - Target directory already exists, skipping creation.


模型已下载到: Qwen3-TTS-CustomVoice-0.6B-fp16-ov


## 选择推理设备
[返回目录 ⬆️](#目录：)

选择用于推理的设备。CPU 兼容性最好，GPU 可在支持的硬件上提供加速。

In [2]:
from notebook_utils import device_widget

device = device_widget("CPU", exclude=["NPU"])
device

Dropdown(description='Device:', options=('CPU', 'GPU', 'AUTO'), value='CPU')

## 加载模型
[返回目录 ⬆️](#目录：)

使用 `OVQwen3TTSModel` 加载 OpenVINO 模型。该 wrapper 提供与原始模型相同的 API 接口。

In [3]:
from qwen_3_tts_helper import OVQwen3TTSModel

ov_model = OVQwen3TTSModel.from_pretrained(
    model_dir=model_dir,
    device=device.value,
)

print(f"\n✅ 模型加载完成!")
print(f"   模型类型: {ov_model.tts_model_type}")
print(f"   模型大小: {ov_model.tts_model_size}")
print(f"   支持的说话人: {ov_model.get_supported_speakers()}")
print(f"   支持的语言: {ov_model.get_supported_languages()}")

✅ Patched torch.diff in masking_utils.find_packed_sequence_indices for OpenVINO compatibility


SoX could not be found!

    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    



********
********
 
No checkpoint_path.txt found, trying to load processor from model_dir
Loading OpenVINO speech tokenizer

✅ 模型加载完成!
   模型类型: custom_voice
   模型大小: 0b6
   支持的说话人: ['aiden', 'dylan', 'eric', 'ono_anna', 'ryan', 'serena', 'sohee', 'uncle_fu', 'vivian']
   支持的语言: ['auto', 'chinese', 'english', 'french', 'german', 'italian', 'japanese', 'korean', 'portuguese', 'russian', 'spanish']


## 运行语音合成
[返回目录 ⬆️](#目录：)

测试 CustomVoice 模式的语音合成。你可以指定不同的说话人和风格指令。

In [4]:
import time
import IPython.display as ipd

test_text = "你好！欢迎来到 OpenVINO Workshop，这是一个语音合成的测试。"

print(f"输入文本: {test_text}")
print("\n正在生成语音...")

start_time = time.time()

wavs, sr = ov_model.generate_custom_voice(
    text=test_text,
    speaker="vivian",
    language="Chinese",
    instruct="用友好亲切的语气说话。",
    non_streaming_mode=True,
    max_new_tokens=2048,
)

inference_time = time.time() - start_time

if wavs is not None:
    audio_duration = len(wavs[0]) / sr
    print(f"\n✅ 语音生成完成!")
    print(f"   推理时间: {inference_time:.2f}s")
    print(f"   音频时长: {audio_duration:.2f}s")
    print(f"   实时因子 (RTF): {inference_time/audio_duration:.3f}")
    print(f"   采样率: {sr} Hz")

    ipd.display(ipd.Audio(wavs[0], rate=sr))

输入文本: 你好！欢迎来到 OpenVINO Workshop，这是一个语音合成的测试。

正在生成语音...

✅ 语音生成完成!
   推理时间: 79.33s
   音频时长: 6.94s
   实时因子 (RTF): 11.436
   采样率: 24000 Hz


## 交互式演示
[返回目录 ⬆️](#目录：)

启动 Gradio 交互式演示，你可以：
- 输入任意文本进行语音合成
- 选择不同的说话人
- 选择语言
- 添加风格指令

In [ ]:
from gradio_helper import make_demo

demo = make_demo(ov_model, model_type=ov_model.tts_model_type)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)

D:\ai-contest\modelscope-workshop\lab3-text-to-speech\gradio_helper.py:203: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme, css=css, title="Qwen3-TTS with OpenVINO") as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


D:\ai-contest\modelscope-workshop\ov_workshop\Lib\site-packages\gradio\processing_utils.py:689: UserWarning: Trying to convert audio automatically from float32 to 16-bit int format.
  warnings.warn(warning.format(data.dtype))
